# Generate optional logarithmic-grid backward diagnostic matrix
Creates `backward_from_log_128.pt`. This file is optional for diagnostics and is not part of the current training forward chain.

In [ ]:
import torch
from differentiable_lensing import DifferentiableLensing
THETA_E_ARCSEC=0.75
LR_PIXEL_SCALE_ARCSEC=0.168
HR_PIXEL_SCALE_ARCSEC=0.084
LR_GRID_SHAPE=64
HR_GRID_SHAPE=128
ALPHA_R_LR=THETA_E_ARCSEC/LR_PIXEL_SCALE_ARCSEC
ALPHA_R_HR=THETA_E_ARCSEC/HR_PIXEL_SCALE_ARCSEC
LOG_C=4.5
DEVICE='cpu'
print(ALPHA_R_LR,ALPHA_R_HR)

In [ ]:
lens=DifferentiableLensing(DEVICE,target_resolution=1.0,target_shape=HR_GRID_SHAPE,alpha=None)
lower,upper=-HR_GRID_SHAPE/2.0,HR_GRID_SHAPE/2.0
_,_,beta_x_log,beta_y_log=lens.make_log_grid(lower,upper,HR_GRID_SHAPE,c=LOG_C)
bx=beta_x_log.unsqueeze(0); by=beta_y_log.unsqueeze(0)
alpha=lens.construct_sis(bx,by,ALPHA_R_HR)
theta_x,theta_y=lens.backward_lensing(bx,by,alpha)
grid_fracs=lens.log_grid_crop(beta_x_log,beta_y_log,theta_x[0],theta_y[0])
areas=lens.nsq_As(theta_x[0],theta_y[0])
M,shape=lens.build_sparse_mapping(grid_fracs,areas,DEVICE)
M=M.coalesce()
assert M.shape==(HR_GRID_SHAPE**2,HR_GRID_SHAPE**2)
assert torch.isfinite(M.values()).all()
torch.save(M,'backward_from_log_128.pt')
print('saved backward_from_log_128.pt',M.shape,M._nnz(),shape)